# Week 8 – Autonomous Deal-Hunting AI

An agentic AI system that scans online deals, estimates fair prices using multiple AI/ML models, and surfaces great opportunities.


In [ ]:
import os
import sys
import json
import pickle
import logging
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)

required_vars = ["OPENAI_API_KEY", "HF_TOKEN"]
optional_vars = ["DEEPSEEK_API_KEY", "PUSHOVER_USER", "PUSHOVER_TOKEN"]

print("Required:")
for v in required_vars:
    print(f"  {v}: {'SET' if os.getenv(v) else 'MISSING'}")
print("Optional:")
for v in optional_vars:
    print(f"  {v}: {'SET' if os.getenv(v) else 'NOT SET'}")

## Phase 1 – Modal: Deploy the fine-tuned SpecialistAgent

**Prerequisites:**
1. Run `modal setup` from the command line to authenticate
2. Create a secret called **hf-secret** in [Modal Secrets](https://modal.com/secrets) with your `HF_TOKEN`
3. Deploy the pricer service: `modal deploy pricer_service.py`

In [ ]:
import modal

modal_config = Path.home() / ".modal.toml"
print(f"Modal version: {modal.__version__}")
print(f"Config found: {modal_config.exists()}")
if not modal_config.exists():
    print("Run 'modal setup' in your terminal first.")

In [ ]:
# Deploy the pricer service (uncomment to run)
# !modal deploy ../../../week8/pricer_service.py

In [ ]:
from agents.specialist_agent import SpecialistAgent

logging.basicConfig(level=logging.INFO, format="%(message)s")

specialist = SpecialistAgent()

test_items = [
    "iPad Pro 2nd generation with 256GB storage",
    "Sony WH-1000XM5 wireless noise-cancelling headphones",
    "Nintendo Switch OLED model with neon controllers",
]

print("\nSpecialistAgent predictions:")
print("=" * 60)
for item in test_items:
    price = specialist.price(item)
    print(f"  {item}")
    print(f"  -> ${price:.2f}")
    print("-" * 60)

## Phase 2 – RAG: Build the FrontierAgent with ChromaDB

We vectorize product data with `all-MiniLM-L6-v2`, store in ChromaDB, and use RAG to give GPT context for pricing.

In [ ]:
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from huggingface_hub import login
import chromadb

hf_token = os.environ["HF_TOKEN"]
login(hf_token, add_to_git_credential=True)

DB = "products_vectorstore"

### Load training data

You need `train.pkl` and `test.pkl` from Week 6.  
Copy them into this folder, or adjust the paths below.

In [ ]:
week8_dir = Path("../../../week8")

for pkl_name in ["train.pkl", "test.pkl"]:
    local = Path(pkl_name)
    parent = week8_dir / pkl_name
    if local.exists():
        path = local
    elif parent.exists():
        path = parent
    else:
        raise FileNotFoundError(f"{pkl_name} not found locally or in {week8_dir}")
    with open(path, "rb") as f:
        globals()[pkl_name.replace(".pkl", "")] = pickle.load(f)
    print(f"Loaded {pkl_name} from {path} ({len(globals()[pkl_name.replace('.pkl', '')]):,} items)")

In [ ]:
def description(item):
    """Extract product description without the pricing question."""
    text = item.prompt.replace("How much does this cost to the nearest dollar?\n\n", "")
    return text.split("\n\nPrice is $")[0]

print(f"Sample: {description(train[0])[:200]}...")

In [ ]:
client = chromadb.PersistentClient(path=DB)
collection_name = "products"

existing = [c.name for c in client.list_collections()]
if collection_name in existing:
    client.delete_collection(collection_name)
    print(f"Deleted existing '{collection_name}' collection")

collection = client.create_collection(collection_name)
print(f"Created collection: {collection_name}")

In [ ]:
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

NUM_DOCS = 20_000

print(f"Vectorizing {NUM_DOCS:,} products...")
for i in tqdm(range(0, NUM_DOCS, 1000)):
    batch = train[i : i + 1000]
    docs = [description(item) for item in batch]
    vecs = encoder.encode(docs).astype(float).tolist()
    metas = [{"category": item.category, "price": item.price} for item in batch]
    ids = [f"doc_{j}" for j in range(i, i + len(docs))]
    collection.add(ids=ids, documents=docs, embeddings=vecs, metadatas=metas)

print(f"Done. {NUM_DOCS:,} products in ChromaDB.")

In [ ]:
query = "Sony wireless noise-cancelling headphones"
qvec = encoder.encode([query]).astype(float).tolist()
results = collection.query(query_embeddings=qvec, n_results=5)

print(f"Query: '{query}'\n")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0]), 1):
    print(f"{i}. ${meta['price']:.2f} | {meta['category']}")
    print(f"   {doc[:120]}...\n")

In [ ]:
from agents.frontier_agent import FrontierAgent

frontier = FrontierAgent(collection)

frontier_test = [
    "Apple iPad Pro 12.9-inch with 256GB storage",
    "Sony WH-1000XM5 wireless headphones 30-hour battery",
    "Nintendo Switch OLED with neon Joy-Con controllers",
]

print("FrontierAgent predictions:")
print("=" * 60)
for item in frontier_test:
    est = frontier.price(item)
    print(f"  {item}")
    print(f"  -> ${est:.2f}")
    print("-" * 60)

## Phase 3 – Train a Random Forest on embeddings

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

result = collection.get(include=["embeddings", "metadatas"])
vectors = np.array(result["embeddings"])
prices = np.array([m["price"] for m in result["metadatas"]])

print(f"Vectors: {vectors.shape}")
print(f"Price range: ${prices.min():.2f} - ${prices.max():.2f}")

In [ ]:
print("Training Random Forest...")
rf_model = RandomForestRegressor(
    n_estimators=100, max_depth=20, random_state=42, n_jobs=-1, verbose=1
)
rf_model.fit(vectors, prices)

train_preds = rf_model.predict(vectors)
mae = np.mean(np.abs(prices - train_preds))
rmse = np.sqrt(mean_squared_error(prices, train_preds))
r2 = r2_score(prices, train_preds)

print(f"\nTraining MAE : ${mae:.2f}")
print(f"Training RMSE: ${rmse:.2f}")
print(f"Training R2  : {r2:.4f}")

In [ ]:
joblib.dump(rf_model, "random_forest_model.pkl")
print(f"Saved random_forest_model.pkl ({os.path.getsize('random_forest_model.pkl') / 1e6:.1f} MB)")

In [ ]:
rf_test = [
    "Wireless Bluetooth headphones with noise cancellation",
    "USB-C laptop charger 65W power adapter",
    "Mechanical gaming keyboard RGB backlit",
]

print("Random Forest direct predictions:")
print("=" * 60)
for item in rf_test:
    vec = encoder.encode([item])
    pred = max(0, rf_model.predict(vec)[0])
    print(f"  {item} -> ${pred:.2f}")

## Phase 4 – EnsembleAgent: Combine all three models

The `EnsembleAgent` calls SpecialistAgent, FrontierAgent, and NeuralNetworkAgent, then combines using learned weights.

In [ ]:
from agents.ensemble_agent import EnsembleAgent

ensemble = EnsembleAgent(collection)

ensemble_test = [
    "Dell XPS 15 laptop with Intel i7 and 16GB RAM",
    "Sony PlayStation 5 console with controller",
    "Bose QuietComfort 45 wireless headphones",
]

print("\nThree-way comparison:")
print("=" * 80)
for item in ensemble_test:
    s = specialist.price(item)
    f = frontier.price(item)
    e = ensemble.price(item)
    print(f"  {item}")
    print(f"    Specialist : ${s:>8.2f}")
    print(f"    Frontier   : ${f:>8.2f}")
    print(f"    Ensemble   : ${e:>8.2f}")
    print("-" * 80)

## Phase 5 – ScannerAgent: Scrape RSS deals

In [ ]:
from agents.deals import ScrapedDeal, DealSelection, Deal, Opportunity
from agents.scanner_agent import ScannerAgent

print("Fetching deals from RSS feeds...")
deals = ScrapedDeal.fetch(show_progress=True)
print(f"\nFetched {len(deals)} deals")

if deals:
    print(f"\nSample deal:\n{'=' * 60}")
    print(deals[0].describe())
    print("=" * 60)

In [ ]:
scanner = ScannerAgent()

print("ScannerAgent is selecting top deals with GPT...\n")
selected = scanner.scan(memory=[])

if selected:
    print(f"Selected {len(selected.deals)} deals:\n")
    for i, deal in enumerate(selected.deals, 1):
        print(f"Deal {i}: ${deal.price:.2f}")
        print(f"  {deal.product_description[:100]}...")
        print(f"  {deal.url}\n")
else:
    print("No deals found.")

## Phase 6 – PlanningAgent: Orchestrate the full pipeline

The PlanningAgent coordinates Scanner -> Ensemble -> Messenger to find the best deal.

In [ ]:
from agents.planning_agent import PlanningAgent

planner = PlanningAgent(collection)
print("PlanningAgent ready.")

In [ ]:
test_deal = Deal(
    product_description="Sony WH-1000XM5 wireless noise-cancelling headphones, premium sound, 30-hour battery",
    price=249.99,
    url="https://example.com/deal",
)

print(f"Testing with: {test_deal.product_description[:70]}...")
print(f"Listed price: ${test_deal.price:.2f}\n")

opp = planner.run(test_deal)

print(f"\nEstimated value : ${opp.estimate:.2f}")
print(f"Discount        : ${opp.discount:.2f}")
print(f"Good deal?      : {'YES' if opp.discount > 50 else 'No'}")

In [ ]:
print("Running full planning cycle...")
print("  1. Scan RSS feeds")
print("  2. Parse deals with GPT")
print("  3. Price each with EnsembleAgent")
print("  4. Surface best opportunity\n")

best = planner.plan(memory=[])

if best:
    print(f"\nBEST OPPORTUNITY:")
    print("=" * 60)
    print(f"Product : {best.deal.product_description[:100]}...")
    print(f"Price   : ${best.deal.price:.2f}")
    print(f"Estimate: ${best.estimate:.2f}")
    print(f"Savings : ${best.discount:.2f}")
    print(f"URL     : {best.deal.url}")
    print("=" * 60)
else:
    print("No opportunities with discount > $50 found.")

## Phase 7 – Gradio UI

In [ ]:
import gradio as gr
from deal_agent_framework import DealAgentFramework

agent_framework = DealAgentFramework()
agent_framework.init_agents_as_needed()

print("Agent framework ready.")

In [ ]:
with gr.Blocks(title="The Price is Right", fill_width=True) as ui:

    gr.Markdown("<h2 style='text-align:center'>The Price is Right &mdash; Deal Hunting AI</h2>")
    gr.Markdown("<p style='text-align:center'>Autonomous agent framework for finding great deals</p>")

    opportunities_table = gr.Dataframe(
        headers=["Description", "Price", "Estimate", "Discount", "URL"],
        label="Opportunities Found",
        wrap=True,
        column_widths=[4, 1, 1, 1, 2],
        row_count=10,
    )

    scan_btn = gr.Button("Run Scan Cycle", variant="primary")
    status_box = gr.Textbox(label="Status", lines=3)

    def run_scan():
        try:
            before = len(agent_framework.memory)
            all_opps = agent_framework.run()
            if all_opps:
                rows = [
                    [
                        opp.deal.product_description[:80] + "...",
                        f"${opp.deal.price:.2f}",
                        f"${opp.estimate:.2f}",
                        f"${opp.discount:.2f}",
                        opp.deal.url,
                    ]
                    for opp in all_opps
                ]
                if len(all_opps) > before:
                    latest = all_opps[-1]
                    msg = f"New opportunity! Discount: ${latest.discount:.2f}"
                else:
                    msg = "Scan complete. No new opportunities (discount < $50)."
                return rows, msg
            return gr.update(), "Scan complete. No opportunities found."
        except Exception as e:
            import traceback
            return gr.update(), f"Error: {e}\n\n{traceback.format_exc()}"

    scan_btn.click(run_scan, outputs=[opportunities_table, status_box])

In [ ]:
ui.launch(inbrowser=True)